# Assignment: Explainable AI for Time Series Forecasting

**Dataset:** Beijing PM2.5 air quality, 2010–2014 (Liang et al., 2015, UCI Machine Learning Repository)

**Name:** ______________________ **Roll No.:** ______________________

In this assignment you build next-day air-pollution forecasters for Beijing and explain them with permutation importance, SHAP, LIME and attribution methods. The lecture notebook used a univariate temperature series. This dataset adds real weather drivers, so you must explain *exogenous* inputs, not just lags.

### Instructions
- Rename this file `XAI_TS_<RollNo>.ipynb` and submit it run top to bottom, with all outputs visible.
- Every `# TODO` cell must be completed. Answer each question in the **Your answer** cell below its code.
- Written answers: 3–6 sentences each, quoting the numbers your code produced.
- Use only data up to day *t − 1* to forecast day *t*, unless a question explicitly says otherwise. Leakage costs marks in every part it affects.
- Keep `SEED = 42` for NumPy, PyTorch, LightGBM and LIME.
- You may reuse functions from the lecture notebook, but you must adapt them.
- The notebook must run on a CPU in under 20 minutes.

### Marks
| Part | Topic | Marks |
|---|---|---|
| A | Exploration and a sanity prior | 10 |
| B | Feature engineering and forecasters | 15 |
| C | SHAP | 25 |
| D | LIME | 15 |
| E | Sequence model and attribution methods | 15 |
| F | Evaluating explanations | 10 |
| G | Reflection | 10 |
| Bonus | Extensions | +10 |
| | **Total** | **100** |

## 0. Setup
Uncomment the install line if any library is missing.

In [ ]:
# !pip install shap lime lightgbm statsmodels torch scikit-learn pandas matplotlib seaborn scipy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

import lightgbm as lgb
import shap
from lime.lime_tabular import LimeTabularExplainer

import torch
import torch.nn as nn

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
print("shap", shap.__version__, "| lightgbm", lgb.__version__, "| torch", torch.__version__)

## Dataset

Hourly PM2.5 concentration at the US Embassy in Beijing, with weather from Beijing Capital Airport: 43,824 hours. You will aggregate to **daily** values (1,826 days) and forecast **tomorrow's mean PM2.5 (µg/m³)**.

| Column | Meaning | Daily aggregation |
|---|---|---|
| `pm2.5` | PM2.5 concentration (µg/m³), **target** | mean |
| `DEWP` | Dew point (°C), a humidity proxy | mean |
| `TEMP` | Temperature (°C) | mean |
| `PRES` | Air pressure (hPa) | mean |
| `cbwd` | Combined wind direction: NW, NE, SE, cv (calm/variable) | most frequent |
| `Iws` | Cumulated wind speed (m/s) | max |
| `Is` | Cumulated hours of snow | max |
| `Ir` | Cumulated hours of rain | max |

**Chronological split (fixed for everyone):** train 2010–2012, validation 2013 (early stopping only), test 2014. Compute every explanation on the 2014 test year.

**Domain hints.** Beijing pollution builds up over several stagnant, humid days and is flushed out by strong, dry north-westerly winds. Winter heating raises baseline levels. These are your *sanity prior*.

The starter code below is provided; run it as given.

In [ ]:
URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pollution.csv"
raw = pd.read_csv(URL)
raw["date"] = pd.to_datetime(raw[["year", "month", "day", "hour"]])
raw = raw.set_index("date").drop(columns=["No", "year", "month", "day", "hour"])

daily = raw.resample("D").agg({
    "pm2.5": "mean", "DEWP": "mean", "TEMP": "mean", "PRES": "mean",
    "Iws": "max", "Is": "max", "Ir": "max",
    "cbwd": lambda s: s.mode().iloc[0] if len(s.mode()) else np.nan,
})
print(daily.shape, "|", daily["pm2.5"].isna().sum(), "days without PM2.5")
daily.head()

In [ ]:
def metrics(y_true, y_pred):
    """MAE and RMSE. Always call this with values in µg/m³."""
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred))}

---
# Part A: Exploration and a sanity prior (10 marks)
Build the prior that your explanations will be checked against later.

### A1 (2 marks)
Handle the 37 days with no PM2.5 reading. Write code that compares two strategies, **linear interpolation** and **forward fill**, and justify the one you keep. Explain why you must never fill gaps using values from the future *when building features*.

In [ ]:
# TODO A1: compare interpolation vs forward fill (e.g. plot both around a long gap,
# compare summary statistics), then create the cleaned series used for the rest of the notebook.
pm_interp = ...
pm_ffill = ...

daily_clean = daily.copy()
# daily_clean["pm2.5"] = ...

**A1 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

### A2 (2 marks)
Plot daily PM2.5 for 2010–2014 with a 30-day rolling mean. Then plot its histogram and that of `log1p(PM2.5)`. Report the skewness of both and decide whether to model the **raw** or the **log** target. State how this choice will change the *units* of every SHAP value you report later.

In [ ]:
# TODO A2: time plot with 30-day rolling mean; two histograms; skewness of both.
USE_LOG_TARGET = ...   # True or False, set after your analysis

**A2 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

### A3 (3 marks)
Run an STL decomposition (period 365) on your chosen target. Report the share of variance in the seasonal and residual components. Compare with the temperature series from the lecture (66.9% seasonal): is pollution more or less predictable from the calendar alone? Explain why physically.

In [ ]:
# TODO A3: STL decomposition and variance shares.

**A3 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

### A4 (3 marks)
1. Plot the ACF and PACF of the target up to lag 30.
2. Compute the Spearman correlation between **tomorrow's** PM2.5 and **today's** value of each weather variable (`DEWP`, `TEMP`, `PRES`, `Iws`, `Is`, `Ir`).
3. Show the mean next-day PM2.5 by today's wind direction (`cbwd`).

From these results, write down your **sanity prior**: a ranked list of the 4–5 inputs you expect a faithful explanation to put at the top, with the expected sign (+ / −) of each.

In [ ]:
# TODO A4.1: ACF / PACF

In [ ]:
# TODO A4.2 and A4.3: correlations with next-day PM2.5; mean next-day PM2.5 by cbwd

**A4 — Your sanity prior:**

| Rank | Input | Expected sign | Reason |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |
| 4 | | | |
| 5 | | | |

---
# Part B: Feature engineering and forecasters (15 marks)

### B1 (5 marks)
Write `make_features(df) -> pd.DataFrame` that creates, for each day *t*, **only information available at the end of day t − 1**:

| Group | Features to create |
|---|---|
| PM2.5 recent lags | `pm_lag_1` … `pm_lag_3` |
| PM2.5 older lags | `pm_lag_4` … `pm_lag_7` |
| PM2.5 rolling stats | `pm_roll_mean_3`, `pm_roll_mean_7`, `pm_roll_std_7` |
| Weather, yesterday | `DEWP_lag_1`, `TEMP_lag_1`, `PRES_lag_1`, `Iws_lag_1`, `Is_lag_1`, `Ir_lag_1` |
| Weather change | `PRES_diff_1` (= PRES at t−1 minus t−2), `DEWP_diff_1` |
| Wind direction | one-hot of `cbwd` at t − 1 (4 columns) |
| Calendar | `doy_sin`, `doy_cos`, `is_heating_season` (15 Nov – 15 Mar), `dow` |

Store the groups in a dict `FEATURE_GROUPS`. Then write the leakage test: overwrite every raw value on one test day with random numbers, rebuild the features, and `assert` that the feature row for that same day is unchanged.

In [ ]:
def make_features(df: pd.DataFrame) -> pd.DataFrame:
    X = pd.DataFrame(index=df.index)
    pm = df["pm2.5"]
    # TODO: PM2.5 lags 1..7
    # TODO: rolling stats computed on pm.shift(1)
    # TODO: yesterday's weather, weather changes, one-hot wind direction at t-1
    # TODO: calendar features (these are known in advance, so day t is fine)
    # Target (transform it if USE_LOG_TARGET)
    X["target"] = ...
    return X.dropna()

FEATURE_GROUPS = {
    "PM recent lags (1-3)": [...],
    "PM older lags (4-7)":  [...],
    "PM rolling stats":     [...],
    "Weather yesterday":    [...],
    "Weather change":       [...],
    "Wind direction":       [...],
    "Calendar":             [...],
}

data = make_features(daily_clean)
feature_names = [c for c in data.columns if c != "target"]
print(f"{len(data)} samples x {len(feature_names)} features")

In [ ]:
# TODO B1 leakage test
test_day = pd.Timestamp("2014-06-15")
corrupted = daily_clean.copy()
# corrupted.loc[test_day, <numeric columns>] = random values
# assert make_features(corrupted).loc[test_day, feature_names].equals(data.loc[test_day, feature_names])
print("Leakage test passed")

### B2 (2 marks)
Apply the fixed split. Print the date range and number of rows of train, validation and test.

In [ ]:
# TODO B2
train = ...
val = ...
test = ...

### B3 (4 marks)
Fit and report test **MAE and RMSE in µg/m³** (even if you model the log target) for:
- (a) persistence: tomorrow = today,
- (b) 7-day mean,
- (c) a LightGBM regressor with early stopping on 2013.

Present the results in one table.

In [ ]:
# TODO B3
lgbm = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.02, random_state=SEED, verbose=-1)
# lgbm.fit(..., eval_set=[...], callbacks=[lgb.early_stopping(100, verbose=False)])

results = pd.DataFrame({
    # "Persistence": metrics(...),
    # "7-day mean":  metrics(...),
    # "LightGBM":    metrics(...),
}).T.round(2)
results

**B3 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

### B4 (4 marks) — the leakage experiment
Train a second LightGBM, `lgbm_oracle`, that additionally receives **day t's** weather (`DEWP`, `PRES`, `Iws`, `cbwd` at *t*). This simulates having a perfect weather forecast. Report its test MAE. In 3–4 sentences, explain when using such features is legitimate in a real system and when it is leakage. You will explain both models in Part C.

In [ ]:
# TODO B4
lgbm_oracle = ...

**B4 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

---
# Part C: SHAP (25 marks)
Use `shap.TreeExplainer` on the LightGBM from B3 unless told otherwise. If you model the log target, state clearly that SHAP values are in log units.

### C1 (3 marks)
Compute SHAP values for the 2014 test set. Verify the **efficiency** property numerically: report the maximum absolute gap between base value + Σφ and the model output. Report the base value and translate it into µg/m³.

In [ ]:
# TODO C1
tree_explainer = shap.TreeExplainer(lgbm)
shap_exp = ...

**C1 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

### C2 (5 marks)
Produce the global **bar** plot and **beeswarm** plot. Compare the top five features with your sanity prior from A4. For each weather feature in the top ten, state whether its colour pattern (direction) matches the physics in the domain hints. Identify one feature whose effect is *conditional*, like `lag_2` in the lecture, and explain it.

In [ ]:
# TODO C2: bar plot

In [ ]:
# TODO C2: beeswarm plot

**C2 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

### C3 (5 marks)
Choose three test days:
1. the most polluted **forecast**,
2. the day with the largest absolute **error**,
3. the day with the largest one-day **drop** in actual PM2.5 (a clean-air event).

Draw a waterfall plot for each. For the clean-air day: did the model anticipate the drop, and which inputs pushed it down? If it missed, what information was it lacking?

In [ ]:
# TODO C3
idx_high = ...
idx_err = ...
idx_drop = ...

**C3 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

### C4 (4 marks)
Draw two dependence plots: `Iws_lag_1` coloured by the NW wind indicator, and `pm_lag_1` coloured by `DEWP_lag_1`. Describe the shape of each learned response curve (linear, saturating, threshold) and the interaction the colour reveals.

In [ ]:
# TODO C4

**C4 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

### C5 (5 marks) — grouped and seasonal SHAP
Sum SHAP values within each group of `FEATURE_GROUPS`. Plot:
- (a) mean |grouped SHAP| as a bar chart,
- (b) a month × group heatmap of mean |grouped SHAP| for 2014.

Is the model mainly a persistence model or a weather model? Does reliance on weather change between the heating and non-heating seasons?

In [ ]:
# TODO C5
group_shap = ...

**C5 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

### C6 (3 marks) — explaining the oracle
Repeat C5(a) for `lgbm_oracle` from B4. How does the importance of the PM2.5 lags change when the model can see day *t*'s weather? Explain what this shows about how correlated inputs share credit.

In [ ]:
# TODO C6

**C6 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

---
# Part D: LIME (15 marks)

In [ ]:
def predict_fn(arr):
    return lgbm.predict(pd.DataFrame(arr, columns=feature_names))

### D1 (4 marks)
Explain the most polluted forecast and the clean-air day from C3 with `LimeTabularExplainer` (5,000 samples). Report the black-box forecast, LIME's surrogate prediction and the local R² for each. Which of the two explanations would you trust more, and why?

In [ ]:
# TODO D1
lime_explainer = LimeTabularExplainer(...)

**D1 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

### D2 (4 marks) — unrealistic neighbours
Generate LIME's perturbation samples for one instance (or reproduce LIME's sampling yourself) and count the fraction of neighbours that are **physically implausible**. Define implausible with at least two rules you justify, for example:
- `pm_lag_1` more than 3 × `pm_roll_mean_3` (impossible by construction),
- snow hours > 0 with `TEMP_lag_1` > 10 °C.

Explain why this matters more for time series than for ordinary tabular data.

In [ ]:
# TODO D2

**D2 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

### D3 (4 marks) — stability
Re-run LIME 15 times with different seeds, using **500** and **5,000** samples, on the same instance. Show a boxplot of the weights for the top eight features and a table of the coefficient of variation. List the features whose **sign** flips across seeds.

In [ ]:
# TODO D3

**D3 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

### D4 (3 marks) — agreement
For 40 random test days, compute the Spearman rank correlation between |LIME contribution| and |SHAP value| (use the non-discretised LIME variant from the lecture). Report the mean and plot the histogram. Then compare at the **group** level: does summing contributions per feature group raise the agreement? Explain the result.

In [ ]:
# TODO D4

**D4 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

---
# Part E: A sequence model and attribution methods (15 marks)

### E1 (3 marks)
Build input windows of **W = 14** days with **six channels** per step: scaled `log1p(PM2.5)`, `DEWP`, `PRES`, `Iws`, `doy_sin`, `doy_cos`. Fit all scalers on the training years only. Train a small LSTM or 1-D CNN with early stopping on 2013, and add its test MAE (µg/m³) to the table from B3.

In [ ]:
W = 14
CHANNELS = ["log_pm", "DEWP", "PRES", "Iws", "doy_sin", "doy_cos"]
# TODO E1: build X_seq (n, W, 6), y_seq, dates; split; scale using training statistics only

class SeqForecaster(nn.Module):
    def __init__(self, n_in=len(CHANNELS), hidden=48):
        super().__init__()
        # TODO
    def forward(self, x):          # x: (batch, W, channels)
        ...                        # TODO: return shape (batch,)

torch.manual_seed(SEED)
model = SeqForecaster()
# TODO: training loop with early stopping on the 2013 windows

**E1 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

### E2 (5 marks)
Implement **from scratch in PyTorch** (no Captum): Saliency, Gradient × Input, SmoothGrad and Integrated Gradients. Use the average training window as the IG baseline. Verify IG **completeness** numerically and report the mean gap. Then compute GradientSHAP with `shap.GradientExplainer` and 200 background windows.

In [ ]:
BASELINE = ...   # (1, W, C) average training window

def _grad(x):
    # TODO: gradient of the summed model output w.r.t. the input
    ...

def saliency(x): ...
def grad_x_input(x): ...
def smoothgrad(x, n=30, sigma=0.15): ...
def integrated_gradients(x, baseline=BASELINE, steps=64): ...

ATTR = {}
# ATTR["Saliency"] = ...
# TODO: completeness check for IG
# TODO: GradientSHAP

**E2 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

### E3 (3 marks)
Implement **Occlusion** (window k = 2 days, all channels) and a **segment-based LIME** (7 segments of 2 days, baseline replacement, weighted ridge).

In [ ]:
def occlusion(x, k=2, baseline=BASELINE): ...
def lime_segments(x, n_seg=7, n_samples=600, kernel_width=0.35, baseline=BASELINE, seed=SEED): ...

# ATTR["Occlusion"] = ...
# ATTR["Segment-LIME"] = ...

### E4 (4 marks)
For the model's largest-error test day, plot:
- (a) the 14-day input history,
- (b) a methods × time-step heatmap of attributions summed over channels, normalised per row,
- (c) an Integrated Gradients **time × channel** heatmap.

Which channel and which day drove that forecast? Does the explanation agree with the domain hints?

> **Baseline question.** The average training window means "a typical fortnight of pollution and weather." Is this a sensible baseline for pollution, given its heavy right skew and strong winter/summer difference? Propose one alternative baseline and say how it would change the meaning of the attributions.

In [ ]:
# TODO E4

**E4 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

---
# Part F: Evaluating explanations (10 marks)

### F1 (3 marks) — agreement
Compute the mean per-instance Spearman correlation between the absolute per-time-step attributions of every pair of methods from Part E. Show it as a heatmap. Which methods form clusters, and why?

In [ ]:
# TODO F1

**F1 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

### F2 (5 marks) — faithfulness
Implement a **deletion** test (Most Relevant First) on the PM2.5 channel *and* an **insertion** test (start from the baseline and add the most relevant steps first). Include a **random** ordering as a control. Report normalised AUCs for every method in one table.

Plot the memory profile (mean |attribution| per time step over 2014, per method). Unlike the temperature model in the lecture, which had a one-day memory, does your test now separate the methods? Explain why or why not.

In [ ]:
def deletion_curve(order, x, baseline=BASELINE, ks=range(0, W + 1)): ...
def insertion_curve(order, x, baseline=BASELINE, ks=range(0, W + 1)): ...
# TODO F2

**F2 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

### F3 (2 marks) — channel-level faithfulness
Repeat the deletion test, but delete one whole **channel** at a time, ranked by each method's total channel attribution. Do the methods agree on whether wind or humidity matters more?

In [ ]:
# TODO F3

**F3 — Your answer:**

*Write 3–6 sentences here, quoting the numbers your code produced.*

---
# Part G: Reflection (10 marks)
Answer in prose, 150–250 words each, citing your own results.

### G1 (4 marks)
You must explain tomorrow's pollution forecast to a city health officer with no ML training. Which single explanation from this assignment would you show them, and what would you say? Which explanation would you *not* show, and why?

**G1 — Your answer:**

### G2 (3 marks)
Name the two design choices that changed your explanations the most (for example target transform, baseline, feature grouping, number of LIME samples). Give the evidence from your notebook.

**G2 — Your answer:**

### G3 (3 marks)
Propose an explanation-based **drift alarm** for this model in production. State exactly what you would compute each week and the threshold you would use, then test it by comparing 2013 with 2014.

In [ ]:
# TODO G3: compute your drift statistic for 2013 vs 2014

**G3 — Your answer:**

---
# Bonus (up to 10 marks): pick up to two

- **Multi-horizon (5).** Forecast PM2.5 at t+1, t+3 and t+7 with three LightGBMs. Show how grouped SHAP shifts from lags to calendar and weather as the horizon grows.
- **TimeSHAP-style pruning (5).** For the sequence model, group all time steps older than *k* days into one "past" coalition and compute KernelSHAP on the reduced input. Find the smallest *k* that keeps 90% of the attribution.
- **Counterfactual (5).** For the most polluted test day, find the smallest change in yesterday's wind speed and direction that brings the forecast below 75 µg/m³ (China's 24-hour PM2.5 standard). Is the counterfactual physically plausible?

In [ ]:
# Bonus (optional)

---
### Declaration
I confirm this notebook is my own work. AI assistants used (if any) and what for:

*...*